Model Caller

In [6]:
import math
import json
import os
import numpy as np
import librosa
import joblib
from tensorflow.keras import models

# --- Global Configuration Constants for Emotion Prediction Model ---
# These constants are used by the 'predict_emotion_chunks' function and other feature extraction steps.
SR = 22050          # Sample rate
N_MELS = 128        # Number of Mel bands for spectrograms
N_FFT = 2048        # FFT window size
HOP_LENGTH = 512    # Hop length for STFT and Mel Spectrograms
SEGMENT_SECS = 10   # Duration of audio segments for individual model predictions (changed from 3 to 10)
SEG_FRAMES = math.ceil(SEGMENT_SECS * SR / HOP_LENGTH) # Number of frames per segment

# --- Emotion Assignment Function ---
# This function assigns an emotion label based on valence and arousal values.
# It is called by the 'predict_emotion_chunks' function.
def assign_emotion(valence, arousal):
    """Assigns an emotion label based on valence and arousal values."""
    if valence >= 0.7 and arousal >= 0.7:
        return "excited"
    elif valence >= 0.5 and arousal >= 0.5:
        return "happy"
    elif valence < 0.5 and arousal >= 0.5:
        return "tense"
    elif valence < 0.3 and arousal < 0.3:
        return "sad"
    elif valence >= 0.5 and arousal < 0.5:
        return "calm"
    else:
        return "neutral"

In [40]:
import librosa
import numpy as np
import pandas as pd
from google.colab import files
import math
import json
import os
import joblib
from tensorflow.keras import models
from sklearn.preprocessing import MinMaxScaler

# ======================
# GLOBAL CONFIGURATION CONSTANTS
# ======================
SR = 22050          # Sample rate
N_MELS = 128        # Number of Mel bands for spectrograms
N_FFT = 2048        # FFT window size
HOP_LENGTH = 512    # Hop length
SEGMENT_SECS = 10   # Model context window
CSV_INTERVAL = 0.2  # Target resolution for the output CSV (Changed to 1 second)
SEG_FRAMES = math.ceil(SEGMENT_SECS * SR / HOP_LENGTH)

# ======================
# MODEL/SCALER PATHS
# ======================
model_path = "best_cnn_model.keras"
scheduler_path = "label_scaler.pkl"

# ======================
# HELPER FUNCTIONS
# ======================
def assign_emotion(valence, arousal):
  if valence >= 0.55 and arousal >= 0.55: return "excited"
  elif valence >= 0.45 and arousal >= 0.45: return "happy"
  elif valence < 0.45 and arousal >= 0.45: return "tense"
  elif valence < 0.40 and arousal < 0.40: return "sad"
  elif valence >= 0.45 and arousal < 0.45: return "calm"
  else: return "neutral"

def load_emotion_artifacts(model_path, scaler_path):
    model = models.load_model(model_path)
    scaler = joblib.load(scaler_path)
    return model, scaler

def detect_major_minor_quality(y_segment, sr_loaded, n_fft=N_FFT, hop_length=HOP_LENGTH):
    if len(y_segment) < n_fft:
        return "Minor" # Default for very short segments (arbitrarily chosen)

    # Compute chroma features (often a prerequisite for tonnetz)
    chroma = librosa.feature.chroma_stft(y=y_segment, sr=sr_loaded, n_fft=n_fft, hop_length=hop_length)

    # If chroma has too few frames for tonnetz, return Minor
    if chroma.shape[1] < 2:
        return "Minor"

    # Compute Tonnetz. librosa.effects.harmonic helps separate harmonic content.
    tonnetz = librosa.feature.tonnetz(y=librosa.effects.harmonic(y_segment), sr=sr_loaded, chroma=chroma)

    # Average over time to get a single Tonnetz vector for the segment
    avg_tonnetz = np.mean(tonnetz, axis=1)

    # The 6th component of Tonnetz (index 5) is strongly associated with major/minor mode.
    # Positive values suggest major, negative values suggest minor.
    major_minor_component = avg_tonnetz[5] if len(avg_tonnetz) >= 6 else 0

    if major_minor_component > 0:
        return "Major"
    else:
        return "Minor"

def get_segment_emotions(audio_path, model, scaler):
    y_audio, sr_loaded = librosa.load(audio_path, sr=SR, mono=True)
    results = []
    seg_len = int(SEGMENT_SECS * sr_loaded)

    for start in range(0, len(y_audio), seg_len):
        segment = y_audio[start : start + seg_len]
        if len(segment) < seg_len:
            segment = np.pad(segment, (0, seg_len - len(segment)))

        mel = librosa.feature.melspectrogram(y=segment, sr=sr_loaded, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH)
        mel_db = librosa.power_to_db(mel, ref=np.max)

        if mel_db.shape[1] < SEG_FRAMES:
            mel_db = np.pad(mel_db, ((0, 0), (0, SEG_FRAMES - mel_db.shape[1])))
        else:
            mel_db = mel_db[:, :SEG_FRAMES]

        X = np.expand_dims(np.expand_dims(mel_db, axis=0), -1)
        preds = model.predict(X, verbose=0)
        vals = scaler.inverse_transform(preds)[0]

        # Detect major/minor quality for the current segment
        key_type = detect_major_minor_quality(segment, sr_loaded)

        results.append({
            "start": start / sr_loaded,
            "end": (start + seg_len) / sr_loaded,
            "valence": vals[0],
            "arousal": vals[1],
            "emotion": assign_emotion(vals[0], vals[1]),
            "key_type": key_type # Add key type to results
        })
    return results

# ======================
# EXECUTION
# ======================
uploaded = files.upload()
if not uploaded:
    print("No file uploaded.")
else:
    file_path = list(uploaded.keys())[0]
    # Extract filename without extension
    base_name = os.path.splitext(file_path)[0]
    output_filename = f"{base_name}.csv"

    model, scaler = load_emotion_artifacts(model_path, scheduler_path)

    y, sr = librosa.load(file_path, sr=SR)
    duration = librosa.get_duration(y=y, sr=sr)

    # Removed tempo calculation
    # overall_tempo, beat_frames = librosa.beat.beat_track(y=y, sr=sr)
    # print(f"Estimated overall tempo: {overall_tempo[0]:.2f} BPM")

    rms = librosa.feature.rms(y=y, hop_length=HOP_LENGTH).flatten()
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=HOP_LENGTH).flatten()
    bass = np.abs(librosa.stft(y))[(librosa.fft_frequencies(sr=sr) < 200)].mean(axis=0)
    treble = np.abs(librosa.stft(y))[(librosa.fft_frequencies(sr=sr) >= 2000)].mean(axis=0)

    def norm(x): return (x - x.min()) / (x.max() - x.min()) if x.max() != x.min() else np.zeros_like(x)
    n_rms, n_bass, n_treble, n_pitch = map(norm, [rms, bass, treble, centroid])
    sig_times = np.arange(len(rms)) * HOP_LENGTH / sr

    emo_chunks = get_segment_emotions(file_path, model, scaler)

    data = []
    target_times = np.arange(0, duration, CSV_INTERVAL)

    for t in target_times:
        sig_idx = np.argmin(np.abs(sig_times - t))
        e_curr = next((c for c in emo_chunks if c["start"] <= t < c["end"]) ,
                      emo_chunks[-1]) # Fallback to the last chunk if no match is found (e.g., end of audio)

        timestamp = f"{int(t//60):02d}:{t%60:06.3f}"

        # Audio Intensity based on normalized RMS
        rms_score = n_rms[sig_idx]
        if rms_score > 0.66:
            audio_intensity = "high"
        elif rms_score > 0.33:
            audio_intensity = "medium"
        else:
            audio_intensity = "low"

        # Sound Fullness based on normalized bass and treble
        sound_fullness_score = (n_bass[sig_idx] + n_treble[sig_idx]) / 2.0
        if sound_fullness_score > 0.6:
            sound_fullness = "Dense"
        elif sound_fullness_score > 0.3:
            sound_fullness = "Round"
        else:
            sound_fullness = "Sparse"

        # Simplified combined mood categories (2x2 grid for energy and valence)
        energy_val = (e_curr["arousal"] + n_rms[sig_idx]) / 2.0
        valence_val = e_curr["valence"]

        valence_threshold = 0.5
        effective_valence_threshold = valence_threshold

        # Determine timbre characteristic based on normalized bass and treble
        timbre_focus = ""
        bass_val = n_bass[sig_idx]
        treble_val = n_treble[sig_idx]

        if bass_val > treble_val * 1.5: # Bass significantly higher
            timbre_focus = "bass-dominant"
        elif treble_val > bass_val * 1.5: # Treble significantly higher
            timbre_focus = "treble-dominant"
        else: # Roughly balanced
            timbre_focus = "balanced timbre"



        # The user has previously requested to remove "intensity" and audio_intensity from combined_mood string.
        combined_mood = assign_mood(valence_val, energy_val, rms[sig_idx], centroid[sig_idx], bass[sig_idx], treble[sig_idx], e_curr["key_type"])

        data.append({
            "time": timestamp,
            "energy": round(float(n_rms[sig_idx]), 3),
            "pitch": round(float(n_pitch[sig_idx]), 3),
            "bass": round(float(n_bass[sig_idx]), 3),
            "treble": round(float(n_treble[sig_idx]), 3),
            "valence": round(e_curr["valence"], 3),
            "arousal": round(e_curr["arousal"], 3),
            "mood": e_curr["emotion"],

        })

    df = pd.DataFrame(data)
    display(df.head(20))
    df.to_csv(output_filename, index=False)
    files.download(output_filename)

Saving APLMate.com - City of Stars - Ryan Gosling_ Emma Stone.mp3 to APLMate.com - City of Stars - Ryan Gosling_ Emma Stone.mp3


,time,energy,pitch,bass,treble,valence,arousal,mood
0,00:00.000,0.000,1.000,0.000,0.000,0.503,0.441,calm
1,00:00.200,0.000,0.488,0.001,0.001,0.503,0.441,calm
2,00:00.400,0.000,0.508,0.000,0.001,0.503,0.441,calm
3,00:00.600,0.133,0.003,0.174,0.002,0.503,0.441,calm
4,00:00.800,0.046,0.018,0.068,0.002,0.503,0.441,calm
5,00:01.000,0.247,0.005,0.293,0.003,0.503,0.441,calm
6,00:01.200,0.181,0.019,0.261,0.006,0.503,0.441,calm
7,00:01.400,0.167,0.024,0.220,0.007,0.503,0.441,calm
8,00:01.600,0.235,0.014,0.401,0.005,0.503,0.441,calm
9,00:01.800,0.110,0.024,0.173,0.005,0.503,0.441,calm


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
!pip install moviepy

In [14]:
import cv2
import numpy as np
import moviepy.editor as mp
from google.colab import files
import os

# 1. Define video parameters
WIDTH, HEIGHT = 1280, 720
FPS = 24
audio_path = 'SuiteNo3.mp3'
video_temp = 'temp_render.mp4'
output_final = 'output_visualization.mp4'

# Get audio duration
audio_clip = mp.AudioFileClip(audio_path)
duration = audio_clip.duration
total_frames = int(duration * FPS)

# Define features to plot
features = ['energy', 'pitch', 'bass', 'treble', 'valence', 'arousal']
colors = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (0, 255, 255), (255, 0, 255), (255, 255, 0)]

# Prepare VideoWriter
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(video_temp, fourcc, FPS, (WIDTH, HEIGHT))

print(f"Generating {total_frames} frames...")

# 2-6. Iterate through frames and draw
for i in range(total_frames):
    t = i / FPS
    # Find nearest row in df based on time
    # df['time'] is in MM:SS.mmm format, need to match with t (seconds)
    # Since CSV_INTERVAL was 1.0, we can approximate index
    idx = min(int(round(t)), len(df) - 1)
    row = df.iloc[idx]

    # Create blank canvas
    frame = np.zeros((HEIGHT, WIDTH, 3), dtype=np.uint8)

    # 4. Draw dynamic bar chart
    bar_width = 150
    gap = 40
    start_x = (WIDTH - (len(features) * bar_width + (len(features) - 1) * gap)) // 2

    for j, feature in enumerate(features):
        val = row[feature]
        bar_height = int(val * 500) # Scale value for visibility
        x1 = start_x + j * (bar_width + gap)
        y1 = HEIGHT - 100 - bar_height
        x2 = x1 + bar_width
        y2 = HEIGHT - 100

        cv2.rectangle(frame, (x1, y1), (x2, y2), colors[j], -1)
        cv2.putText(frame, feature.capitalize(), (x1, HEIGHT - 70),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(frame, f"{val:.2f}", (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)

    # 5. Add persistent mood overlay (top-right)
    mood_text = f"Mood: {row['mood'].upper()}"
    cv2.putText(frame, mood_text, (WIDTH - 400, 100),
                cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255, 255, 255), 3)

    # Add timestamp
    cv2.putText(frame, f"Time: {t:.2f}s", (50, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (200, 200, 200), 2)

    out.write(frame)

out.release()

# 7. Merge audio and video using MoviePy
print("Merging audio...")
video_clip = mp.VideoFileClip(video_temp)
final_video = video_clip.set_audio(audio_clip)
final_video.write_videofile(output_final, codec='libx264', audio_codec='aac', fps=FPS)

# 8. Provide download
files.download(output_final)
print("Done!")

Generating 8665 frames...
Merging audio...
Moviepy - Building video output_visualization.mp4.
MoviePy - Writing audio in output_visualizationTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video output_visualization.mp4



Moviepy - Done !
Moviepy - video ready output_visualization.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done!


**Reasoning**:
I will rewrite the rendering script as instructed to incorporate multiprocessing for frame generation and optimize the MoviePy assembly process for faster performance.



In [17]:
import cv2
import numpy as np
import moviepy.editor as mp
from google.colab import files
import os
from multiprocessing import Pool
import shutil

# 1. Configuration and Pre-calculated Geometries
WIDTH, HEIGHT = 1280, 720
FPS = 24
AUDIO_PATH = 'SuiteNo3.mp3'
TEMP_FRAME_DIR = 'temp_frames'
VIDEO_TEMP = 'fast_temp_render.mp4'
OUTPUT_FINAL = 'optimized_visualization.mp4'

if os.path.exists(TEMP_FRAME_DIR):
    shutil.rmtree(TEMP_FRAME_DIR)
os.makedirs(TEMP_FRAME_DIR)

# Global-like access for multiprocessing
FEATURES = ['energy', 'pitch', 'bass', 'treble', 'valence', 'arousal']
COLORS = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (0, 255, 255), (255, 0, 255), (255, 255, 0)]

# Pre-calculated geometries
BAR_WIDTH = 150
GAP = 40
START_X = (WIDTH - (len(FEATURES) * BAR_WIDTH + (len(FEATURES) - 1) * GAP)) // 2
BAR_BASE_Y = HEIGHT - 100

def render_frame(frame_idx):
    t = frame_idx / FPS
    # CSV_INTERVAL is 0.2, so index calculation is t / 0.2
    idx = min(int(round(t / 0.2)), len(df) - 1)
    row = df.iloc[idx]

    frame = np.zeros((HEIGHT, WIDTH, 3), dtype=np.uint8)

    # Draw Bars
    for j, feature in enumerate(FEATURES):
        val = row[feature]
        bar_height = int(val * 500)
        x1 = START_X + j * (BAR_WIDTH + GAP)
        y1 = BAR_BASE_Y - bar_height
        x2 = x1 + BAR_WIDTH
        y2 = BAR_BASE_Y

        cv2.rectangle(frame, (x1, y1), (x2, y2), COLORS[j], -1)
        cv2.putText(frame, feature.capitalize(), (x1, HEIGHT - 70),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(frame, f"{val:.2f}", (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)

    # Add Mood and Time
    mood_text = f"Mood: {str(row['mood']).upper()}"
    cv2.putText(frame, mood_text, (WIDTH - 400, 100),
                cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255, 255, 255), 3)
    cv2.putText(frame, f"Time: {t:.2f}s", (50, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (200, 200, 200), 2)

    frame_path = os.path.join(TEMP_FRAME_DIR, f"frame_{frame_idx:06d}.jpg")
    cv2.imwrite(frame_path, frame)
    return frame_path

# 2. Parallel Rendering
audio_clip = mp.AudioFileClip(AUDIO_PATH)
duration = audio_clip.duration
total_frames = int(duration * FPS)

print(f"Starting parallel rendering of {total_frames} frames...")
with Pool(os.cpu_count()) as pool:
    pool.map(render_frame, range(total_frames))

# 3. Assemble Frames into Video
print("Assembling video from frames...")
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
video_writer = cv2.VideoWriter(VIDEO_TEMP, fourcc, FPS, (WIDTH, HEIGHT))

for i in range(total_frames):
    img_path = os.path.join(TEMP_FRAME_DIR, f"frame_{i:06d}.jpg")
    img = cv2.imread(img_path)
    if img is not None:
        video_writer.write(img)
video_writer.release()

# 4. Final Muxing with MoviePy
print("Final muxing with audio...")
video_clip = mp.VideoFileClip(VIDEO_TEMP)
final_video = video_clip.set_audio(audio_clip)
final_video.write_videofile(
    OUTPUT_FINAL,
    codec='libx264',
    audio_codec='aac',
    fps=FPS,
    threads=os.cpu_count(),
    preset='ultrafast'
)

files.download(OUTPUT_FINAL)
print("Optimization Complete.")

Starting parallel rendering of 8665 frames...
Assembling video from frames...
Final muxing with audio...
Moviepy - Building video optimized_visualization.mp4.
MoviePy - Writing audio in optimized_visualizationTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video optimized_visualization.mp4



Moviepy - Done !
Moviepy - video ready optimized_visualization.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Optimization Complete.
